# ClusterBP v2: exploratory data analysis

Descriptive statistics of the modelling cohort (patients with at least one prediction window, 9,413 in the
operator's export) for the paper:

1. distributions of SBP and DBP and the share of readings ≥ 140, 160 and 180 mm Hg;
2. measurements per patient and per day, and their distribution over the time of day (patient's local time);
3. share of windows followed by an event of type A or B, by month and by region;
4. recurrence of events: share of patients with at least one A or B event, share of events in patients who already
   had an event of this type, median interval between events.

Events of one type of a patient on one day are merged into an episode (as in `clusterbp_v2_additional.ipynb`).
95% CIs: patient-level bootstrap (a patient is resampled with all their measurements, windows or episodes).

Output: `results_v2/eda/eda_summary.csv` and `results_v2/eda/eda.png`. Before writing, the current `results_v2/`
is copied once to `results_v2_before_eda_baselines/`.

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from clusterbp_lib import RAW_DIR, DATA_DIR, RESULTS_DIR, SEED, TARGETS, check_windows, load_regions
from clusterbp_extra import snapshot_results, N_BOOT

warnings.filterwarnings('ignore')
snapshot_results()
OUT = os.path.join(RESULTS_DIR, 'eda')
os.makedirs(OUT, exist_ok=True)
EDA_TARGETS = ['A', 'B']

check_windows()
win = pd.read_parquet(os.path.join(DATA_DIR, 'windows.parquet'),
                      columns=['pid', 'cut'] + [f'y_{t}' for t in EDA_TARGETS] + [f'ev_{t}_days_since' for t in EDA_TARGETS])
cohort = np.unique(win['pid'].values)
meas = pd.read_parquet(os.path.join(DATA_DIR, 'measurements.parquet'))
meas = meas[meas['pid'].isin(cohort)].reset_index(drop=True)
events = pd.read_parquet(os.path.join(DATA_DIR, 'events.parquet'))
events = events[events['pid'].isin(cohort)].reset_index(drop=True)
print(f'Cohort: {len(cohort):,} patients, {len(meas):,} measurements, {len(win):,} windows, {len(events):,} events')

rows = []


def add(section, metric, value, group='', lo=np.nan, hi=np.nan, n=np.nan):
    rows.append({'section': section, 'metric': metric, 'group': group, 'value': value,
                 'ci_low': lo, 'ci_high': hi, 'n': n})


def boot_ratio(num, den, pid, n_boot=N_BOOT, seed=SEED):
    """Patient-level bootstrap of Σnum / Σden: 95% CI from resampling patients with all their rows."""
    u, pidx = np.unique(pid, return_inverse=True)
    sn = np.bincount(pidx, num.astype(float), len(u))
    sd = np.bincount(pidx, den.astype(float), len(u))
    rng = np.random.default_rng(seed)
    est = np.empty(n_boot)
    for b in range(n_boot):
        w = np.bincount(rng.integers(0, len(u), len(u)), minlength=len(u))
        est[b] = (w * sn).sum() / max((w * sd).sum(), 1e-12)
    return sn.sum() / sd.sum(), np.percentile(est, 2.5), np.percentile(est, 97.5)


def boot_patient_stat(values_by_patient, stat, n_boot=N_BOOT, seed=SEED):
    """Patient-level bootstrap of stat(concatenated values) for a list of per-patient arrays."""
    rng = np.random.default_rng(seed)
    n = len(values_by_patient)
    est = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.integers(0, n, n)
        est[b] = stat(np.concatenate([values_by_patient[i] for i in pick]))
    return stat(np.concatenate(values_by_patient)), np.percentile(est, 2.5), np.percentile(est, 97.5)

## 1. Blood pressure distributions

In [ ]:
mp = meas['pid'].values
for col, name in [('sbp', 'SBP'), ('dbp', 'DBP')]:
    v = meas[col].values
    q = np.percentile(v, [25, 50, 75])
    add('BP distribution', f'{name}, mean', v.mean(), n=len(v))
    add('BP distribution', f'{name}, SD', v.std(ddof=1), n=len(v))
    add('BP distribution', f'{name}, median [IQR]', q[1], lo=q[0], hi=q[2], n=len(v))
for thr in [140, 160, 180]:
    est, lo, hi = boot_ratio(meas['sbp'].values >= thr, np.ones(len(meas)), mp)
    add('BP distribution', f'share of readings with SBP ≥ {thr}', est, lo=lo, hi=hi, n=len(meas))
for thr in [90, 100, 110]:
    est, lo, hi = boot_ratio(meas['dbp'].values >= thr, np.ones(len(meas)), mp)
    add('BP distribution', f'share of readings with DBP ≥ {thr}', est, lo=lo, hi=hi, n=len(meas))
pd.DataFrame(rows)

## 2. Measurement frequency and time of day

- **per patient:** number of measurements during follow-up;
- **per day:** measurements per patient-day with at least one measurement, and measurements per day of follow-up
  (number of measurements / days from the first to the last measurement);
- **time of day:** the hour of the measurement in the patient's local time. The export has the measurement time and
  the patient's time zone. Three readings of the measurement time are checked (as recorded; UTC; Moscow time) and
  the one with the fewest night readings (00:00–04:59) is used; all three are printed below.

In [ ]:
g = meas.assign(day=np.floor(meas['td'])).groupby('pid')
per_pat = g.size()
span = g['td'].agg(lambda s: np.floor(s.max()) - np.floor(s.min()) + 1)
per_day_fu = per_pat / span
per_active_day = meas.assign(day=np.floor(meas['td'])).groupby(['pid', 'day']).size()
for metric, s in [('measurements per patient', per_pat), ('follow-up, days', span),
                  ('measurements per day of follow-up (per patient)', per_day_fu),
                  ('measurements per patient-day with measurements', per_active_day),
                  ('share of follow-up days with a measurement (per patient)',
                   per_active_day.groupby(level=0).size() / span)]:
    q = np.percentile(s.values, [25, 50, 75])
    add('measurement frequency', f'{metric}: median [IQR]', q[1], lo=q[0], hi=q[2], n=len(s))
    add('measurement frequency', f'{metric}: mean', s.mean(), n=len(s))


def tz_hours(s):
    """Patient's UTC offset in hours from the time-zone column (numbers, '+03:00', 'UTC+3', 'GMT+7' or IANA names)."""
    from zoneinfo import ZoneInfo
    out = {}
    for v in pd.unique(s.astype(str)):
        h = np.nan
        if '/' in v:
            try:
                off = pd.Timestamp('2024-10-01', tz=ZoneInfo(v.strip())).utcoffset()
                h = off.total_seconds() / 3600
            except Exception:
                pass
        else:
            m = re.search(r'([+-]?)\s*(\d{1,2})(?::?(\d{2}))?', v)
            if m:
                h = int(m.group(2)) + (int(m.group(3)) / 60 if m.group(3) else 0)
                h = -h if m.group(1) == '-' else h
        out[v] = h
    return s.astype(str).map(out).astype(float)


raw = pd.read_csv(os.path.join(RAW_DIR, 'Первичные данные от 10.04.2025.csv'),
                  usecols=['id пациента', 'время измерения', 'часовой пояс пациента'], low_memory=False)
raw = raw.rename(columns={'id пациента': 'pid', 'время измерения': 't_meas', 'часовой пояс пациента': 'tz'})
raw['pid'] = pd.to_numeric(raw['pid'], errors='coerce')
raw = raw[raw['pid'].isin(cohort)].drop_duplicates(['pid', 't_meas'])
t = pd.to_datetime(raw['t_meas'], errors='coerce')
tz = tz_hours(raw['tz'])
print(f'Time zone parsed for {tz.notna().mean():.1%} of measurements; values: '
      f'{tz.value_counts().head(8).round(2).to_dict()}')
hour_raw = t.dt.hour + t.dt.minute / 60
readings = {'as recorded': hour_raw % 24,
            'UTC → local': (hour_raw + tz) % 24,
            'Moscow time (UTC+3) → local': (hour_raw + tz - 3) % 24}
night = {k: ((h >= 0) & (h < 5)).mean() for k, h in readings.items()}
choice = min(night, key=night.get)
print('Share of readings at 00:00–04:59 under each reading of the measurement time:',
      {k: round(v, 3) for k, v in night.items()}, '→ using:', choice)
hour = readings[choice].dropna()
hpid = raw.loc[hour.index, 'pid'].values
add('time of day', 'time convention used', np.nan, group=choice)
for label, a, b in [('night 00:00–05:59', 0, 6), ('morning 06:00–11:59', 6, 12),
                    ('afternoon 12:00–17:59', 12, 18), ('evening 18:00–23:59', 18, 24)]:
    est, lo, hi = boot_ratio(((hour >= a) & (hour < b)).values, np.ones(len(hour)), hpid)
    add('time of day', f'share of readings, {label}', est, lo=lo, hi=hi, n=len(hour))
hour_counts = np.bincount(np.floor(hour.values).astype(int) % 24, minlength=24)
pd.DataFrame(rows).tail(14)

## 3. Share of windows followed by an event, by month and by region

Month of the window's cutoff. The first and last months of the export are only partly covered by cutoffs
(e.g. November 2023 and January 2025); they are marked `complete month = False`, kept in the table and left out
of the figure.

In [ ]:
regions = load_regions()
win['month'] = pd.to_datetime(win['cut'] * 86400, unit='s').dt.to_period('M').astype(str)
win['region'] = regions.reindex(win['pid'].values).fillna('not specified').values
cut_days = pd.to_datetime(win['cut'] * 86400, unit='s').dt.normalize()
first_cut, last_cut = cut_days.min(), cut_days.max()


def complete_month(m):
    p = pd.Period(m, 'M')
    return bool(p.start_time.normalize() >= first_cut and p.end_time.normalize() <= last_cut)


print(f'Cutoffs from {first_cut:%Y-%m-%d} to {last_cut:%Y-%m-%d}; incomplete months:',
      [m for m in sorted(win['month'].unique()) if not complete_month(m)])
by_rows = []
for t in EDA_TARGETS:
    y = win[f'y_{t}'].values
    est, lo, hi = boot_ratio(y, np.ones(len(y)), win['pid'].values)
    add('event rate', f'share of windows followed by a type-{t} event', est, lo=lo, hi=hi, n=len(y))
    for by in ['month', 'region']:
        for lev, idx in win.groupby(by).indices.items():
            est, lo, hi = boot_ratio(y[idx], np.ones(len(idx)), win['pid'].values[idx])
            full = complete_month(lev) if by == 'month' else True
            by_rows.append({'target': t, 'by': by, 'group': lev, 'rate': est, 'lo': lo, 'hi': hi,
                            'windows': len(idx), 'patients': len(np.unique(win['pid'].values[idx])),
                            'complete month': full if by == 'month' else np.nan})
            add(f'event rate by {by}', f'share of windows followed by a type-{t} event'
                + ('' if full else ' (incomplete month)'), est, group=lev, lo=lo, hi=hi, n=len(idx))
by_df = pd.DataFrame(by_rows)
by_df[by_df['by'] == 'month']

## 4. Recurrence of events

- share of cohort patients with at least one episode of the type during follow-up;
- share of episodes that occurred in patients who already had an episode of this type (all episodes except each
  patient's first);
- share of positive windows in which the patient already had an event of this type before the cutoff (the
  "repetition" the models can use);
- median interval between consecutive episodes of the same patient, days.

In [ ]:
for t in EDA_TARGETS:
    e = events[events['code'].isin(TARGETS[t]['codes'])]
    ep = e.assign(day=np.floor(e['td'])).groupby(['pid', 'day']).size().reset_index()[['pid', 'day']]
    has = np.isin(cohort, ep['pid'].unique()).astype(float)
    est, lo, hi = boot_ratio(has, np.ones(len(cohort)), cohort)
    add('recurrence', f'share of patients with ≥ 1 {t} episode', est, lo=lo, hi=hi, n=len(cohort))
    first = ep.groupby('pid')['day'].transform('min') == ep['day']
    est, lo, hi = boot_ratio(~first.values, np.ones(len(ep)), ep['pid'].values)
    add('recurrence', f'share of {t} episodes in patients with an earlier {t} episode', est, lo=lo, hi=hi, n=len(ep))
    add('recurrence', f'{t} episodes per patient with ≥ 1 episode: median', ep.groupby('pid').size().median(),
        n=ep['pid'].nunique())
    yw = win[f'y_{t}'].values == 1
    prior = win[f'ev_{t}_days_since'].notna().values
    est, lo, hi = boot_ratio(prior[yw], np.ones(yw.sum()), win['pid'].values[yw])
    add('recurrence', f'share of {t}-positive windows with an earlier {t} event', est, lo=lo, hi=hi, n=int(yw.sum()))
    gaps = [np.diff(np.sort(d.values)) for _, d in ep.groupby('pid')['day'] if len(d) > 1]
    if gaps:
        est, lo, hi = boot_patient_stat(gaps, np.median)
        q = np.percentile(np.concatenate(gaps), [25, 75])
        add('recurrence', f'interval between consecutive {t} episodes, days: median', est, lo=lo, hi=hi,
            n=int(sum(len(x) for x in gaps)))
        add('recurrence', f'interval between consecutive {t} episodes, days: IQR', np.nan, lo=q[0], hi=q[1])

summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(OUT, 'eda_summary.csv'), index=False)
by_df.to_csv(os.path.join(OUT, 'eda_event_rate_by_month_region.csv'), index=False)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 70)
display(summary[summary['section'].isin(['BP distribution', 'measurement frequency', 'time of day', 'recurrence',
                                         'event rate'])].round(4))

## Figure

In [ ]:
COL = {'A': '#3b6ea8', 'B': '#d9822b'}
fig, ax = plt.subplots(2, 2, figsize=(14, 10), gridspec_kw={'width_ratios': [1, 1.15]})

a = ax[0, 0]
bins = np.arange(39.5, 252.5, 2)       # BP is recorded in whole mm Hg: bin edges between integers, 2 mm Hg per bin
a.hist(meas['sbp'], bins=bins, alpha=0.65, color='#3b6ea8', label='SBP')
a.hist(meas['dbp'], bins=bins, alpha=0.65, color='#5aa469', label='DBP')
for thr in [140, 160, 180]:
    a.axvline(thr, color='k', ls=':', lw=0.9)
    a.text(thr, 1.01, str(thr), transform=a.get_xaxis_transform(), ha='center', va='bottom', fontsize=8)
shares = '\n'.join(f'SBP ≥ {thr}: {(meas["sbp"] >= thr).mean():.1%}' for thr in [140, 160, 180])
a.text(0.97, 0.97, shares, transform=a.transAxes, ha='right', va='top', fontsize=8,
       bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='0.7'))
a.set_xlabel('mm Hg')
a.set_ylabel('measurements')
a.set_title('a) Home BP readings', loc='left', pad=14)
a.legend(fontsize=8, loc='upper left')

a = ax[0, 1]
a.bar(np.arange(24), hour_counts / hour_counts.sum(), color='#7f7f7f', width=0.85)
a.set_xticks(range(0, 24, 3))
a.set_xlabel(f"hour of day (patient's local time; {choice})")
a.set_ylabel('share of measurements')
a.set_title(f'b) Time of day; median {per_pat.median():.0f} measurements per patient, '
            f'{per_active_day.median():.0f} per measurement day', loc='left', fontsize=10)

a = ax[1, 0]
for t in EDA_TARGETS:
    d = by_df[(by_df.target == t) & (by_df.by == 'month') & (by_df['complete month'] == True)].sort_values('group')
    x = np.arange(len(d))
    a.plot(x, d['rate'], marker='o', ms=3, color=COL[t], label=f'target {t}')
    a.fill_between(x, d['lo'], d['hi'], color=COL[t], alpha=0.2)
    a.set_xticks(x)
    a.set_xticklabels(d['group'], rotation=60, ha='right', fontsize=7)
a.set_ylabel('share of windows followed by an event')
a.set_title('c) Event rate by month (95% CI; complete months only)', loc='left')
a.legend(fontsize=8)

a = ax[1, 1]
reg = by_df[(by_df.by == 'region')]
order = reg[reg.target == 'B'].sort_values('rate')['group'].tolist()
yy = np.arange(len(order))
for i, t in enumerate(EDA_TARGETS):
    d = reg[reg.target == t].set_index('group').reindex(order)
    a.errorbar(d['rate'], yy + (i - 0.5) * 0.3, xerr=[d['rate'] - d['lo'], d['hi'] - d['rate']], fmt='o', ms=3,
               color=COL[t], capsize=2, label=f'target {t}')
n_pat = reg[reg.target == 'B'].set_index('group')['patients']
a.set_yticks(yy)
a.set_yticklabels([f'{r} (n={n_pat[r]})' for r in order], fontsize=7)
a.set_xlabel('share of windows followed by an event')
a.set_title('d) Event rate by region (95% CI)', loc='left')
a.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(OUT, 'eda.png'), dpi=200, bbox_inches='tight')
plt.show()